# Generate Radiology Reports with MedGemma 4B IT

I ran the fine-tuning experiment in Google Colab. This notebook can fine-tune MedGemma 1.5 4B IT with the prepared radiology data, then generate a submission from `test.csv`. Accept the model terms on Hugging Face before loading it. A T4 GPU or better is recommended for report generation.

## Install dependencies

Run this cell once in a fresh Colab runtime.

In [ ]:
# %pip install --upgrade --quiet accelerate bitsandbytes transformers huggingface_hub pandas datasets peft trl google-genai pydantic python-dotenv

## Authenticate with Hugging Face

In Colab, create an enabled secret named `HF_TOKEN`. Locally, this cell opens the Hugging Face login flow when no cached token is available.

In [1]:
import os
import sys

google_colab = 'google.colab' in sys.modules and not os.environ.get('VERTEX_PRODUCT')

if google_colab:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
else:
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

/Users/banti/Documents/Work/Agentic AI/ai_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load MedGemma 4B IT

The official quick start uses the `image-text-to-text` pipeline for `medgemma-4b-it`, even when the input contains text only. 4-bit loading is enabled by default for a T4-friendly setup.

In [ ]:
import torch
from transformers import BitsAndBytesConfig, GenerationConfig, pipeline

MODEL_ID = 'google/medgemma-1.5-4b-it'
USE_QUANTIZATION = True
BATCH_SIZE = 3
DO_SAMPLE = False
TEMPERATURE = 0.2
TOP_P = 0.9
REPETITION_PENALTY = 1.08
NO_REPEAT_NGRAM_SIZE = 4

# The base model is loaded by the fine-tuned adapter cell below.
# Keep a high context ceiling to avoid Transformers' default max_length warning.
GENERATION_CONFIG = GenerationConfig(
    do_sample=DO_SAMPLE,
    repetition_penalty=REPETITION_PENALTY,
    no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    max_length=8192,
)
if DO_SAMPLE:
    GENERATION_CONFIG.temperature = TEMPERATURE
    GENERATION_CONFIG.top_p = TOP_P
print(f'Generation: do_sample={DO_SAMPLE}, no response-specific token limit')

/Users/banti/Documents/Work/Agentic AI/ai_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded google/medgemma-1.5-4b-it
Generation: do_sample=False, max_new_tokens=1024


## Fine-tune MedGemma with QLoRA

I ran this fine-tuning step in Google Colab. Run these cells after the model-loading cell to train a QLoRA adapter before generating reports. They use the prepared `train.jsonl` file for training and `test.jsonl` for validation. An A100 is recommended; a T4 may not have enough memory for training.

In [ ]:
from pathlib import Path

from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForImageTextToText, AutoProcessor
from trl import SFTConfig, SFTTrainer

TRAINING_DATA_DIR = Path('processed_data')
if not TRAINING_DATA_DIR.exists():
    TRAINING_DATA_DIR = Path('medGemma/processed_data')
ADAPTER_DIR = Path('medgemma-radiology-qlora')
TRAIN_MAX_LENGTH = 2048
TRAIN_EPOCHS = 2
TRAIN_BATCH_SIZE = 1
TRAIN_GRADIENT_ACCUMULATION_STEPS = 8
TRAIN_LEARNING_RATE = 2e-4

if not torch.cuda.is_available():
    raise RuntimeError('QLoRA fine-tuning requires a CUDA GPU in Colab.')

if 'pipe' in globals():
    del pipe
    torch.cuda.empty_cache()

train_data = load_dataset(
    'json',
    data_files={
        'train': str(TRAINING_DATA_DIR / 'train.jsonl'),
        'validation': str(TRAINING_DATA_DIR / 'test.jsonl'),
    },
)
print(train_data)

use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_storage=compute_dtype,
)

training_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=compute_dtype,
    quantization_config=quantization_config,
    device_map='auto',
    attn_implementation='eager',
)
training_processor = AutoProcessor.from_pretrained(MODEL_ID)
training_processor.tokenizer.padding_side = 'right'
training_model.config.use_cache = False
training_model = prepare_model_for_kbit_training(training_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules='all-linear',
    task_type='CAUSAL_LM',
)
training_model = get_peft_model(training_model, lora_config)
training_model.print_trainable_parameters()


def collate_radiology_examples(examples):
    full_texts = []
    prompt_texts = []
    for example in examples:
        messages = example['messages']
        full_texts.append(
            training_processor.apply_chat_template(
                messages,
                add_generation_prompt=False,
                tokenize=False,
            ).strip()
        )
        prompt_texts.append(
            training_processor.apply_chat_template(
                messages[:-1],
                add_generation_prompt=True,
                tokenize=False,
            ).strip()
        )

    batch = training_processor(
        text=full_texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=TRAIN_MAX_LENGTH,
    )
    prompt_tokens = training_processor(
        text=prompt_texts,
        padding=False,
        truncation=True,
        max_length=TRAIN_MAX_LENGTH,
    )['input_ids']
    labels = batch['input_ids'].clone()
    labels[batch['attention_mask'] == 0] = -100
    for row_number, prompt_ids in enumerate(prompt_tokens):
        prompt_length = min(len(prompt_ids), labels.shape[1])
        labels[row_number, :prompt_length] = -100
    batch['labels'] = labels
    return batch


training_args = SFTConfig(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=TRAIN_GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=TRAIN_LEARNING_RATE,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=use_bf16,
    fp16=not use_bf16,
    max_length=TRAIN_MAX_LENGTH,
    remove_unused_columns=False,
    dataset_kwargs={'skip_prepare_dataset': True},
    report_to='none',
)

trainer = SFTTrainer(
    model=training_model,
    args=training_args,
    train_dataset=train_data['train'],
    eval_dataset=train_data['validation'],
    processing_class=training_processor,
    data_collator=collate_radiology_examples,
)

checkpoints = sorted(
    ADAPTER_DIR.glob('checkpoint-*'),
    key=lambda path: int(path.name.rsplit('-', 1)[-1]),
)
resume_checkpoint = checkpoints[-1] if checkpoints else None
if resume_checkpoint:
    print(f'Resuming fine-tuning from {resume_checkpoint}')
else:
    print('No checkpoint found; starting fine-tuning from the base model.')

trainer.train(
    resume_from_checkpoint=str(resume_checkpoint) if resume_checkpoint else None,
)
trainer.save_model(str(ADAPTER_DIR))
training_processor.save_pretrained(str(ADAPTER_DIR))
print(f'Saved LoRA adapter to {ADAPTER_DIR}')

In [ ]:
import gc

del trainer
del training_model
del training_processor
gc.collect()
torch.cuda.empty_cache()

finetuned_base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=compute_dtype,
    quantization_config=quantization_config,
    device_map='auto',
    attn_implementation='eager',
)
finetuned_model = PeftModel.from_pretrained(finetuned_base, ADAPTER_DIR)
finetuned_processor = AutoProcessor.from_pretrained(ADAPTER_DIR)
finetuned_processor.tokenizer.padding_side = 'left'

pipe = pipeline(
    'image-text-to-text',
    model=finetuned_model,
    processor=finetuned_processor,
)
GENERATION_CONFIG = GenerationConfig(
    do_sample=DO_SAMPLE,
    repetition_penalty=REPETITION_PENALTY,
    no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    max_length=8192,
)
if DO_SAMPLE:
    GENERATION_CONFIG.temperature = TEMPERATURE
    GENERATION_CONFIG.top_p = TOP_P
print(f'Loaded fine-tuned adapter from {ADAPTER_DIR}')
print(f'Generation: do_sample={DO_SAMPLE}, no response-specific token limit')

SyntaxError: invalid syntax (57914556.py, line 30)

## Load the competition inputs and define the report prompt

In [ ]:
import json
from pathlib import Path
import re
import time

import pandas as pd

INPUT_CSV = Path('../test.csv')
OUTPUT_CSV = Path('../final_submission_medgemma.csv')
FAILURES_CSV = Path('../medgemma_failures.csv')
MAX_RETRIES = 6
INITIAL_RETRY_DELAY_SECONDS = 30
MAX_RETRY_DELAY_SECONDS = 240

SYSTEM_INSTRUCTION = '''You are a radiologist preparing a final report.

Core reporting rules:
1. Treat the Normal template as the starting report.
2. Route every dictated finding to the matching FINDINGS field.
3. Replace or modify the corresponding normal statement when the dictation describes an abnormality.
4. Preserve template statements for routinely visualized regions not mentioned in the dictation.
5. Update IMPRESSION to summarize the important abnormal findings.
6. Never add findings unsupported by the dictation or template_content. Metadata is context only.
7. Preserve the supplied template wording and report order whenever possible.
8. Use OTHER FINDINGS only when the template contains that field.
9. Do not repeat the reference example, prompt, or explanations.
10. Produce the shortest medically complete report possible, targeting fewer than 850 tokens.

The output must contain only the required FINDINGS fields and one concise IMPRESSION. Stop immediately after the final sentence of IMPRESSION.'''

required_columns = {
    'case_id', 'modality', 'body_part', 'study_description',
    'patient_age_band', 'patient_sex', 'template_content', 'dictation'
}
test = pd.read_csv(INPUT_CSV, encoding='utf-8-sig', dtype={'case_id': str})
missing = required_columns - set(test.columns)
if missing:
    raise ValueError(f'{INPUT_CSV} is missing columns: {sorted(missing)}')
if test['case_id'].duplicated().any():
    raise ValueError('case_id values must be unique')

TRAINING_DATA_PATH = Path('processed_data/train.jsonl')
if not TRAINING_DATA_PATH.exists():
    TRAINING_DATA_PATH = Path('medGemma/processed_data/train.jsonl')

few_shot_by_modality = {}
with TRAINING_DATA_PATH.open(encoding='utf-8') as handle:
    for line in handle:
        example = json.loads(line)
        modality = str(example.get('metadata', {}).get('modality', '')).strip().casefold()
        if not modality:
            continue
        messages = example['messages']
        candidate = {
            'modality': example['metadata']['modality'],
            'case': messages[1]['content'],
            'report': messages[2]['content'],
        }
        current = few_shot_by_modality.get(modality)
        if current is None or len(candidate['report']) < len(current['report']):
            few_shot_by_modality[modality] = candidate


def make_few_shot_example(modality):
    example = few_shot_by_modality.get(str(modality).strip().casefold())
    if example is None:
        return ''
    return f'''REFERENCE EXAMPLE FOR MODALITY {example['modality']} ONLY.
Do not copy its case-specific findings.

EXAMPLE TEMPLATE AND DICTATION:
{example['case']}

EXAMPLE FINAL REPORT:
{example['report']}'''


def make_user_prompt(row):
    few_shot_example = make_few_shot_example(row['modality'])
    reference_section = (
        f'{few_shot_example}\n\n'
        if few_shot_example
        else 'No training example is available for this modality; solve the current case directly.\n\n'
    )
    return f'''{reference_section}--- CURRENT CASE ---
STUDY CONTEXT (context only; do not invent findings from it):
Modality: {row['modality']}
Body part: {row['body_part']}
Study description: {row['study_description']}
Patient age band: {row['patient_age_band']}
Patient sex: {row['patient_sex']}

NORMAL TEMPLATE:
{row['template_content'].strip()}

DICTATION:
{row['dictation'].strip()}

Return only the shortest medically complete report for the CURRENT CASE. Target fewer than 850 tokens. Do not copy case-specific findings from the reference example.'''

print(f'Loaded {len(test)} cases from {INPUT_CSV}')
print(f'Loaded {len(few_shot_by_modality)} shortest modality-specific examples from {TRAINING_DATA_PATH}')

In [ ]:
def make_messages(row):
    return [
        {
            'role': 'system',
            'content': [{'type': 'text', 'text': SYSTEM_INSTRUCTION}],
        },
        {
            'role': 'user',
            'content': [{'type': 'text', 'text': make_user_prompt(row)}],
        },
    ]


def extract_response_text(output):
    if isinstance(output, list):
        output = output[0]
    generated = output.get('generated_text', output)
    if isinstance(generated, list):
        assistant_messages = [
            message for message in generated
            if isinstance(message, dict) and message.get('role') == 'assistant'
        ]
        message = assistant_messages[-1] if assistant_messages else generated[-1]
        content = message.get('content', '') if isinstance(message, dict) else message
    else:
        content = generated
    if isinstance(content, list):
        content = ''.join(
            part.get('text', '') if isinstance(part, dict) else str(part)
            for part in content
        )
    return str(content).strip()


def generate_reports(rows):
    """Generate raw drafts; Gemini performs final correction and validation."""
    messages = [make_messages(row) for row in rows]
    retry_delay = INITIAL_RETRY_DELAY_SECONDS

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            outputs = pipe(
                text=messages,
                batch_size=len(messages),
                generate_kwargs={'generate_config': GENERATION_CONFIG},
            )
            if len(outputs) != len(rows):
                raise ValueError(
                    f'Model returned {len(outputs)} outputs for {len(rows)} inputs.'
                )
            reports = [extract_response_text(output) for output in outputs]
            if any(not report for report in reports):
                raise ValueError('MedGemma returned an empty output.')
            return reports
        except Exception as error:
            if attempt == MAX_RETRIES:
                case_ids = ', '.join(str(row.case_id) for row in rows)
                raise RuntimeError(
                    f'MedGemma batch failed after {MAX_RETRIES} attempts '
                    f'for cases {case_ids}: {error}'
                ) from error
            case_ids = ', '.join(str(row.case_id) for row in rows)
            print(
                f'Batch [{case_ids}] failed attempt {attempt}: {error}. '
                f'Retrying in {retry_delay}s.'
            )
            time.sleep(retry_delay)
            retry_delay = min(MAX_RETRY_DELAY_SECONDS, retry_delay * 2)


def save_checkpoint():
    test[['case_id', 'report']].to_csv(OUTPUT_CSV, index=False)


def save_failures(failures):
    pd.DataFrame(failures, columns=['case_id', 'error']).to_csv(FAILURES_CSV, index=False)

## Generate the resumable submission

The checkpoint is written after each successful case. Re-running this cell resumes from `final_submission_medgemma.csv` and retries only missing reports.

In [ ]:
test['report'] = pd.NA
if OUTPUT_CSV.exists():
    previous = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig', dtype={'case_id': str})
    if {'case_id', 'report'} <= set(previous.columns):
        saved_reports = previous.drop_duplicates('case_id').set_index('case_id')['report']
        test['report'] = test['case_id'].map(saved_reports)
        print(f'Resuming: {test.report.notna().sum()}/{len(test)} reports already saved.')

failures = []
pending = list(test.index[test['report'].isna()])
already_done = len(test) - len(pending)
started_at = time.monotonic()

print(
    f'Status: {already_done}/{len(test)} already complete. '
    f'{len(pending)} MedGemma requests remaining.'
)

for batch_number, batch_start in enumerate(range(0, len(pending), BATCH_SIZE), start=1):
    batch_indices = pending[batch_start:batch_start + BATCH_SIZE]
    rows = [test.loc[index] for index in batch_indices]
    completed_before = test['report'].notna().sum()
    remaining_before = len(test) - completed_before
    case_ids = ', '.join(str(row.case_id) for row in rows)

    print(
        f'\nBatch {batch_number}/{(len(pending) + BATCH_SIZE - 1) // BATCH_SIZE} starting '
        f'| total done: {completed_before}/{len(test)} '
        f'| remaining: {remaining_before} '
        f'| batch size: {len(rows)} '
        f'| case_ids: {case_ids}',
        flush=True,
    )

    try:
        reports = generate_reports(rows)
        for index, report in zip(batch_indices, reports):
            test.loc[index, 'report'] = report
        save_checkpoint()

        completed_now = test['report'].notna().sum()
        remaining_now = len(test) - completed_now
        elapsed_minutes = (time.monotonic() - started_at) / 60
        rate = (completed_now - already_done) / elapsed_minutes if elapsed_minutes else 0
        eta_minutes = remaining_now / rate if rate else 0

        print(
            f'✓ Done | total: {completed_now}/{len(test)} '
            f'| left: {remaining_now} '
            f'| rate: {rate:.2f} reports/min '
            f'| ETA: ~{eta_minutes:.1f} min',
            flush=True,
        )
    except Exception as error:
        for row in rows:
            failures.append({'case_id': row.case_id, 'error': str(error)})
        save_failures(failures)
        print(
            f'✗ Failed | case_ids: {case_ids} '
            f'| saved error to {FAILURES_CSV}',
            flush=True,
        )

save_checkpoint()
if failures:
    save_failures(failures)
done = test['report'].notna().sum()
left = len(test) - done
print(f'\nFinal status: {done}/{len(test)} complete; {left} left.')
if failures or left:
    raise RuntimeError(
        f'Incomplete submission: {done}/{len(test)} reports generated. '
        f'Check {FAILURES_CSV} and re-run to resume.'
    )
print(f'Finished successfully: {OUTPUT_CSV}')

## Validate the final submission

In [ ]:
submission = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig', dtype={'case_id': str})
if list(submission.columns) != ['case_id', 'report']:
    raise ValueError(f'Unexpected submission columns: {list(submission.columns)}')
if (
    len(submission) != len(test)
    or submission['case_id'].duplicated().any()
    or submission['report'].isna().any()
):
    raise ValueError('Submission validation failed.')
print(submission.head(2).to_string(index=False))
print(f'Validated {len(submission)} rows: {OUTPUT_CSV}')

## Gemini correction and structured rewrite

This optional second stage treats the MedGemma report as an untrusted draft. Gemini may correct wording and formatting, but the template and dictation are the only sources of clinical facts.

In [ ]:
import json
import os
import random
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field


class GeminiReportSchema(BaseModel):
    FINDINGS: str = Field(
        description='Detailed observations using only the supplied template and dictation.'
    )
    IMPRESSION: str = Field(
        description='Concise summary of supported abnormal findings; do not add new information.'
    )


load_dotenv()
gemini_api_key = os.getenv('GEMINI_API_KEY')
if not gemini_api_key and 'google.colab' in __import__('sys').modules:
    from google.colab import userdata
    gemini_api_key = userdata.get('GEMINI_API_KEY')
if not gemini_api_key:
    raise RuntimeError('Set GEMINI_API_KEY in Colab Secrets or the environment.')

GEMINI_MODEL = 'gemini-3.5-flash-lite'
GEMINI_OUTPUT_CSV = Path('final_submission_gemini_refined.csv')
GEMINI_FAILURES_CSV = Path('gemini_refinement_failures.csv')
GEMINI_MAX_RETRIES = 6
GEMINI_REQUESTS_PER_MINUTE = 5
GEMINI_MAX_OUTPUT_TOKENS = 1600

gemini_client = genai.Client(api_key=gemini_api_key)

GEMINI_SYSTEM_INSTRUCTION = '''You are a meticulous radiologist and report editor.

Your task is to correct and compress an untrusted MedGemma draft for the CURRENT CASE.
The supplied template and dictation are the only sources of clinical facts.

Rules:
1. Treat the template as the starting report and preserve its field order and wording when possible.
2. Route every dictated finding to the matching template field.
3. Replace a normal template statement only when the dictation supports an abnormality.
4. Preserve unmentioned template statements, but do not invent details from modality, demographics, or medical knowledge.
5. The MedGemma draft is untrusted: use it only as a formatting/editing hint. Remove unsupported, contradictory, duplicated, hallucinated, or uncertain findings.
6. Correct spelling, malformed headings, repetition, and truncation. Reconstruct the report from the template and dictation when the draft is incomplete.
7. Keep FINDINGS concise and organized by the template fields. Keep IMPRESSION concise and include only conclusions directly supported by the dictation and final FINDINGS.
8. Do not copy findings from the reference example into the CURRENT CASE.
9. Return exactly two strings, FINDINGS and IMPRESSION, with no preamble, explanation, markdown, confidence statement, or extra fields.
10. Target fewer than 850 tokens in the combined report while retaining all clinically relevant dictated findings.'''

GEMINI_CONFIG = types.GenerateContentConfig(
    system_instruction=GEMINI_SYSTEM_INSTRUCTION,
    temperature=0,
    max_output_tokens=GEMINI_MAX_OUTPUT_TOKENS,
    response_mime_type='application/json',
    response_schema=GeminiReportSchema,
)

last_gemini_request_started_at = 0.0
minimum_gemini_request_interval = 60 / GEMINI_REQUESTS_PER_MINUTE


def wait_for_gemini_slot():
    global last_gemini_request_started_at
    remaining = minimum_gemini_request_interval - (
        time.monotonic() - last_gemini_request_started_at
    )
    if remaining > 0:
        time.sleep(remaining)
    last_gemini_request_started_at = time.monotonic()


def gemini_retry_after_seconds(error):
    match = re.search(r'retry[^0-9]*(\d+(?:\.\d+)?)', str(error), flags=re.IGNORECASE)
    return float(match.group(1)) if match else 0.0


def make_gemini_rewrite_prompt(row):
    draft = '' if pd.isna(row.get('report')) else str(row['report'])
    return f'''CURRENT CASE (the only case to report):
Modality: {row['modality']}
Body part: {row['body_part']}
Study description: {row['study_description']}
Patient age band: {row['patient_age_band']}
Patient sex: {row['patient_sex']}

SUPPLIED TEMPLATE:
{row['template_content'].strip()}

DICTATION (authoritative source of findings):
{row['dictation'].strip()}

UNTRUSTED MEDGEMMA DRAFT:
{draft}

Rewrite only the CURRENT CASE. Return the corrected, concise JSON report now.'''


def response_to_gemini_report(response):
    payload = json.loads(response.text)
    parsed = GeminiReportSchema.model_validate(payload)
    findings = parsed.FINDINGS.strip()
    impression = parsed.IMPRESSION.strip()
    if not findings or not impression:
        raise ValueError('Gemini returned an empty FINDINGS or IMPRESSION field.')
    return f'FINDINGS:\n{findings}\n\nIMPRESSION:\n{impression}'


def refine_one_report(row):
    prompt = make_gemini_rewrite_prompt(row)
    retry_delay = 30
    for attempt in range(1, GEMINI_MAX_RETRIES + 1):
        try:
            wait_for_gemini_slot()
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=GEMINI_CONFIG,
            )
            return response_to_gemini_report(response)
        except Exception as error:
            if attempt == GEMINI_MAX_RETRIES:
                raise RuntimeError(
                    f'Gemini refinement failed after {GEMINI_MAX_RETRIES} attempts: {error}'
                ) from error
            wait_seconds = min(
                240,
                max(retry_delay, gemini_retry_after_seconds(error)),
            )
            wait_seconds += random.uniform(0, min(5, wait_seconds * 0.1))
            print(
                f'Gemini case {row.case_id}: attempt {attempt} failed. '
                f'Retrying in {wait_seconds:.1f}s.'
            )
            time.sleep(wait_seconds)
            retry_delay = min(240, retry_delay * 2)

print(f'Gemini refinement model: {GEMINI_MODEL}')

In [ ]:
gemini_test = pd.read_csv(INPUT_CSV, encoding='utf-8-sig', dtype={'case_id': str})
raw_medgemma = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig', dtype={'case_id': str})
if list(raw_medgemma.columns) != ['case_id', 'report']:
    raise ValueError(f'Unexpected MedGemma columns: {list(raw_medgemma.columns)}')

gemini_test = gemini_test.merge(
    raw_medgemma.rename(columns={'report': 'medgemma_report'}),
    on='case_id',
    how='left',
    validate='one_to_one',
)
gemini_test['report'] = pd.NA
if GEMINI_OUTPUT_CSV.exists():
    previous = pd.read_csv(
        GEMINI_OUTPUT_CSV,
        encoding='utf-8-sig',
        dtype={'case_id': str},
    )
    if {'case_id', 'report'} <= set(previous.columns):
        saved_reports = previous.drop_duplicates('case_id').set_index('case_id')['report']
        gemini_test['report'] = gemini_test['case_id'].map(saved_reports)

required_refinement_columns = {
    'case_id', 'modality', 'body_part', 'study_description',
    'patient_age_band', 'patient_sex', 'template_content', 'dictation',
    'medgemma_report', 'report',
}
missing_refinement_columns = required_refinement_columns - set(gemini_test.columns)
if missing_refinement_columns:
    raise ValueError(f'Missing refinement columns: {sorted(missing_refinement_columns)}')


def save_gemini_checkpoint():
    gemini_test[['case_id', 'report']].to_csv(GEMINI_OUTPUT_CSV, index=False)


def save_gemini_failures(failures):
    pd.DataFrame(failures, columns=['case_id', 'error']).to_csv(
        GEMINI_FAILURES_CSV,
        index=False,
    )


failures = []
pending = list(gemini_test.index[gemini_test['report'].isna()])
print(
    f'Gemini refinement status: {len(gemini_test) - len(pending)}/{len(gemini_test)} '
    f'already complete; {len(pending)} remaining.'
)

for request_number, index in enumerate(pending, start=1):
    row = gemini_test.loc[index]
    print(
        f'Gemini request {request_number}/{len(pending)} | case_id: {row.case_id}',
        flush=True,
    )
    try:
        gemini_test.loc[index, 'report'] = refine_one_report(row)
        save_gemini_checkpoint()
        print(
            f'Saved Gemini refinement {gemini_test.report.notna().sum()}/{len(gemini_test)}',
            flush=True,
        )
    except Exception as error:
        failures.append({'case_id': row.case_id, 'error': str(error)})
        save_gemini_failures(failures)
        print(f'Gemini failed for {row.case_id}: {error}', flush=True)

save_gemini_checkpoint()
if failures:
    save_gemini_failures(failures)

done = gemini_test['report'].notna().sum()
left = len(gemini_test) - done
print(f'Gemini refinement complete: {done}/{len(gemini_test)}; {left} left.')
if failures or left:
    raise RuntimeError(
        f'Incomplete Gemini refinement: {done}/{len(gemini_test)} complete. '
        f'Check {GEMINI_FAILURES_CSV} and rerun this cell.'
    )
print(f'Finished refined submission: {GEMINI_OUTPUT_CSV}')

In [ ]:
refined_submission = pd.read_csv(
    GEMINI_OUTPUT_CSV,
    encoding='utf-8-sig',
    dtype={'case_id': str},
)
if list(refined_submission.columns) != ['case_id', 'report']:
    raise ValueError(f'Unexpected refined columns: {list(refined_submission.columns)}')
if (
    len(refined_submission) != len(test)
    or refined_submission['case_id'].duplicated().any()
    or refined_submission['report'].isna().any()
):
    raise ValueError('Gemini refined submission validation failed.')
print(refined_submission.head(2).to_string(index=False))
print(f'Validated refined submission: {GEMINI_OUTPUT_CSV}')